# 03 — Learned-embedding model (PyTorch)

after encoding residues in baseline model as static vectors of AA properties and finding it did not improve metrics,swap to small NN model with learned embeddings as vectors

the vectors start random and are improved via backpropagation by the nn

everything else mirrors the baseline model (data, negative-space expansion,
leakage-safe split by peptide, spectral-angle + pearson) so the number here is directly comparable to the trees

> **Runtime** scales with `FRAC` (cell 2). CPU only
> `FRAC=0.05` ≈ 3 min · `FRAC=0.2` ≈ 10 min · `FRAC=1.0` ≈ 30–60 min.


In [2]:
import os, re, time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn

torch.manual_seed(42)
torch.set_num_threads(os.cpu_count() or 4)   # use all CPU cores
print("torch", torch.__version__, "| threads", torch.get_num_threads())

ModuleNotFoundError: No module named 'torch'

## 1. load & parse data
same load / enzyme parsing / base-peak normalization as `02-model.ipynb`

In [ ]:
df = pd.read_parquet("data_for_student.parquet").merge(
    pd.read_parquet("metadata_for_student.parquet"), on=["raw_file", "scan"], how="left"
)

enzyme_map = ["LysC", "chymo", "LysN", "tryps", "GluC"]
df["enzyme"] = df["raw_file"].apply(
    lambda r: next((n for n in enzyme_map if n in r), "Unknown")
)

# Base-peak normalization: scale each spectrum to its most intense peak -> [0, 1].
df["intensity"] = df["intensity"].apply(
    lambda x: [i / max(x) for i in x] if len(x) > 0 and max(x) > 0 else [0.0] * len(x)
)
print(f"Loaded {len(df):,} spectra")

## 2. Downsample by peptide + expand the negative space
Whole peptides are kept together (no train/test leakage). Every theoretically
possible ECD fragment that was *not* observed gets intensity 0.

In [ ]:
FRAC = 0.05   # fraction of peptides. 0.05 ~3 min, 0.2 ~10 min, 1.0 ~30-60 min.

sampled = df["modified_sequence"].drop_duplicates().sample(frac=FRAC, random_state=42)
df = df[df["modified_sequence"].isin(set(sampled))].reset_index(drop=True)
df["clean_sequence"] = df["modified_sequence"].str.replace(r"\[.*?\]", "", regex=True)
df["charge"] = df["charge"].fillna(0).astype(int)   # precursor charge; 0 = unknown

ION_TYPES = [("C", 1), ("z", 2), ("Z", 3)]


def explode_with_negatives(df):
    cols = ["spectrum_id", "sequence", "enzyme", "ion_type_encoded", "fragment_number",
            "ion_charge", "precursor_charge", "pep_length", "relative_position", "intensity"]
    rows = []
    for sid, seq, clean, ions, intens, enz, pc in zip(
        df.index, df["modified_sequence"], df["clean_sequence"], df["matched_ions"],
        df["intensity"], df["enzyme"], df["charge"]):
        if len(ions) == 0:
            continue
        observed = dict(zip(ions, intens))
        charges = sorted({int(i.split("^")[1]) if "^" in i else 1 for i in ions})
        L = len(clean)
        for ion_type, type_enc in ION_TYPES:
            for fn in range(1, L):                     # cleavage sites 1..L-1
                for charge in charges:
                    ion = f"{ion_type}{fn}" if charge == 1 else f"{ion_type}{fn}^{charge}"
                    rows.append((sid, seq, enz, type_enc, fn, charge, pc, L,
                                 fn / L, observed.get(ion, 0.0)))
    return pd.DataFrame(rows, columns=cols)


ion_df = explode_with_negatives(df)
n_obs = int((ion_df["intensity"] > 0).sum())
print(f"FRAC={FRAC}  rows={len(ion_df):,}  (observed {n_obs:,} | negatives {len(ion_df)-n_obs:,})")

## 3. Residue window (integer IDs)
The 4 residues around the cleavage site `(i-2, i-1, i, i+1)` as integer identities.
Out-of-window positions get `-1`; the embedding cell below remaps that to a padding row.

In [ ]:
AA_INDEX = {aa: idx for idx, aa in enumerate("ACDEFGHIKLMNPQRSTVWY")}

clean_seq = ion_df["sequence"].str.replace(r"\[.*?\]", "", regex=True)
i = ion_df["fragment_number"].values
n = clean_seq.str.len().values
clean_arr = clean_seq.values


def get_residue_vec(clean_arr, pos_arr, length_arr):
    res = np.full(len(clean_arr), -1, dtype=int)
    for j, (s, p, l) in enumerate(zip(clean_arr, pos_arr, length_arr)):
        if 0 <= p < l:
            res[j] = AA_INDEX.get(s[p], -1)
    return res


WINDOW = {"m2": i - 2, "m1": i - 1, "0": i, "p1": i + 1}
for pos_name, pos_arr in WINDOW.items():
    ion_df[f"res_{pos_name}"] = get_residue_vec(clean_arr, pos_arr, n)

RES_ID_COLS = ["res_m2", "res_m1", "res_0", "res_p1"]
NUM_COLS = ["ion_type_encoded", "fragment_number", "ion_charge",
            "pep_length", "relative_position", "precursor_charge"]
print("residue cols:", RES_ID_COLS)
print("numeric cols:", NUM_COLS)

## 4. Leakage-safe split (by peptide, stratified by enzyme)
70 / 15 / 15, identical logic to `02`.

In [ ]:
y = ion_df["intensity"].to_numpy(dtype=float)

seq_tab = ion_df.drop_duplicates("sequence")
seqs = seq_tab["sequence"].to_numpy(dtype=object)
enz = seq_tab["enzyme"].to_numpy(dtype=object)

train_seqs, temp_seqs, _, temp_enz = train_test_split(
    seqs, enz, test_size=0.30, random_state=42, stratify=enz)
val_seqs, test_seqs = train_test_split(
    temp_seqs, test_size=0.50, random_state=42, stratify=temp_enz)
train_seqs, val_seqs, test_seqs = set(train_seqs), set(val_seqs), set(test_seqs)

train_mask = ion_df["sequence"].isin(train_seqs).to_numpy()
val_mask = ion_df["sequence"].isin(val_seqs).to_numpy()
test_mask = ion_df["sequence"].isin(test_seqs).to_numpy()
print(f"train {train_mask.sum():,} | val {val_mask.sum():,} | test {test_mask.sum():,} rows")

## 5. Evaluation functions
SA and pearson, same as in 02

In [ ]:
def spectral_angle(t, p):
    b = np.linalg.norm(t) * np.linalg.norm(p)
    if b < 1e-9:
        return 0.0
    return 1.0 - (2.0 / np.pi) * np.arccos(np.clip(np.dot(t, p) / b, -1.0, 1.0))


def calculate_pearson(t, p):
    if np.std(t) < 1e-9 or np.std(p) < 1e-9:
        return np.nan
    return np.corrcoef(t, p)[0, 1]


MIN_IONS = 5   # tiny spectra make SA / Pearson degenerate

## 6. The embedding model

`nn.Embedding(21, 8)` gives each of the 20 amino acids (plus one padding row for
out-of-window positions) a learned 8-D vector. The 4 window residues are looked
up, flattened, concatenated with the standardized numeric features, and passed
through a small MLP. Trained with Adam + MSE and early-stopped on the validation
loss. MLP = einfachstes NN

In [ ]:
PAD_IDX = 20        # out-of-window residues (-1) map here -> 21 embedding rows
EMB_DIM = 8

# make_inputs macht pandas spalten zu tensors (= wie ein array, aber n-dimensional)
# numerische features werden standardized, nns trainieren schlecht wenn inputs
# sehr unterschiedlich groß sind (frag_number = 20 vs rel_pos = 0.5)
def make_inputs(mask, mean=None, std=None):
    res = ion_df.loc[mask, RES_ID_COLS].to_numpy(np.int64)
    res[res < 0] = PAD_IDX
    num = ion_df.loc[mask, NUM_COLS].to_numpy(np.float32)
    if mean is None:                                  # standardize on TRAIN stats only
        mean, std = num.mean(0), num.std(0) + 1e-6
    num = (num - mean) / std
    return (torch.from_numpy(res), torch.from_numpy(num),
            torch.from_numpy(y[mask].astype(np.float32)), mean, std)


res_tr, num_tr, y_tr, mu, sd = make_inputs(train_mask)
res_va, num_va, y_va, *_ = make_inputs(val_mask, mu, sd)
res_te, num_te, y_te, *_ = make_inputs(test_mask, mu, sd)

# hier ist das NN
class IntensityNet(nn.Module):
    def __init__(self, n_aa=21, emb_dim=EMB_DIM, n_numeric=len(NUM_COLS),
                 hidden=64, n_pos=len(RES_ID_COLS)):
        super().__init__()
        self.emb = nn.Embedding(n_aa, emb_dim)         # the learned AA vectors
        self.mlp = nn.Sequential(
            nn.Linear(n_pos * emb_dim + n_numeric, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    # self.emb(res) holt gelernten 8d vector for jedes der 4 residues
    # flatten(1) legt vektoren hintereinander -> 32 zahlen
    # torch.cat([e, num]) hängt noch numerische features dran -> 38 zahlen
    # self.mlp schiebt 38 zahlen durch NN, -> 1 zahl output = predicted intensity
    def forward(self, res, num):
        e = self.emb(res).flatten(1)    
        return self.mlp(torch.cat([e, num], dim=1)).squeeze(1)

net = IntensityNet()
print(net)
print("trainable params:", sum(p.numel() for p in net.parameters()))

## 6b. Train
Mini-batch Adam, early stopping on validation MSE (patience 5).

In [ ]:
# jede epoch (= ein full pass über alle rows) werden rows geshufflet, gebatcht in 4096
# jede batch: predict, mse (= mean squared error) messen, .backward() callen 
# .backward(): berechnet gradienten -> welche richtung reduziert error rate?
# adam: coacht das model, entscheidet nach MSE wie gewichte verändert werden müssen
# kein improvement for 5 epochs -> early stopping
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
BATCH, MAX_EPOCHS, PATIENCE = 4096, 40, 5

N = res_tr.shape[0]
best_val, best_state, wait = float("inf"), None, 0
t0 = time.time()
for epoch in range(MAX_EPOCHS):
    net.train()
    perm = torch.randperm(N)
    for s in range(0, N, BATCH):
        idx = perm[s:s + BATCH]
        opt.zero_grad()
        loss_fn(net(res_tr[idx], num_tr[idx]), y_tr[idx]).backward()
        opt.step()
    net.eval()
    with torch.no_grad():
        val = loss_fn(net(res_va, num_va), y_va).item()
    print(f"epoch {epoch + 1:2d}  val MSE = {val:.5f}")
    if val < best_val - 1e-5:
        best_val = val
        best_state = {k: v.clone() for k, v in net.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            break
net.load_state_dict(best_state)
print(f"\ntrained {epoch + 1} epochs in {time.time() - t0:.0f}s  (best val MSE = {best_val:.5f})")

## 7. Evaluate — same per-spectrum SA / PCC as the trees

In [ ]:
net.eval()
with torch.no_grad():
    nn_pred = net(res_te, num_te).clamp_min(0).numpy()   # intensities are >= 0

res_df = ion_df.loc[test_mask, ["spectrum_id"]].copy()
res_df["actual"] = y[test_mask]
res_df["pred"] = nn_pred

rows = [(spectral_angle(g["actual"].values, g["pred"].values),
         calculate_pearson(g["actual"].values, g["pred"].values))
        for _, g in res_df.groupby("spectrum_id") if len(g) >= MIN_IONS]
nn_sa = np.array([r[0] for r in rows])
nn_pc = np.array([r[1] for r in rows])

print(f"spectra used: {len(rows)}")
print(f"Embedding NN — median Spectral Angle:      {np.median(nn_sa):.4f}")
print(f"Embedding NN — median Pearson Correlation: {np.nanmedian(nn_pc):.4f}")
print("(Joel's MultiFrag ECD reference: ~0.855 average Pearson)")

## 8. The learned amino-acid embedding

model was only told intensities, never chemistry (=> learned embedding). if
chemically similar residues cluster here, the network discovered that on its own. 

In [ ]:
from sklearn.decomposition import PCA

AA_ORDER = "ACDEFGHIKLMNPQRSTVWY"
vecs = net.emb.weight.detach().numpy()[:20]          # drop padding row (index 20)
xy = PCA(n_components=2, random_state=42).fit_transform(vecs)

BASIC = set("RKH")
fig, ax = plt.subplots(figsize=(6, 6))
for (px, py), aa in zip(xy, AA_ORDER):
    ax.scatter(px, py, color=("crimson" if aa in BASIC else "steelblue"), s=45, zorder=2)
    ax.annotate(aa, (px, py), xytext=(4, 4), textcoords="offset points", fontsize=11)
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_title("Learned amino-acid embedding (PCA)\nred = basic (R / K / H)")
plt.tight_layout()
plt.savefig("fig_embedding.png", dpi=150)
plt.show()